# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

A content page should be prioritized for refresh when it satisfies three conditions:

1. The page receives a high number of Google Search impressions, meaning it still has visibility.
2. The page ranks outside the top search positions, indicating there is room for improvement.
3. The page has not been updated for a long period, making it more likely to contain outdated information.

The baseline score combines these three signals using a transparent rule instead of a machine learning model.

Higher scores indicate higher priority for manual review and content refresh.

## Reason Code

Reason Code:
REFRESH_STALE_HIGH_IMPRESSIONS

Action:
REFRESH_CONTENT

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
from sklearn.preprocessing import MinMaxScaler
import os
import pandas as pd
import numpy as np

df = pd.read_csv("content_refresh_anonymized.csv")
print(df.columns.tolist())


['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [7]:

df = pd.read_csv("content_refresh_anonymized.csv")

features = [
    "impressions_90d",
    "avg_position",
    "days_since_last_update"
]

df = df[df["avg_position"] > 0].copy()

# Normalize features
scaler = MinMaxScaler()

df[[
    "imp_norm",
    "pos_norm",
    "stale_norm"
]] = scaler.fit_transform(df[features])

# Baseline score
df["baseline_score"] = (
      0.50 * df["imp_norm"]
    + 0.30 * df["pos_norm"]
    + 0.20 * df["stale_norm"]
)


df["reason_code"] = "REFRESH_STALE_HIGH_IMPRESSIONS"
df["action"] = "REFRESH_CONTENT"

# Rank
ranked = df.sort_values(
    "baseline_score",
    ascending=False
)

os.makedirs("work/outputs", exist_ok=True)

ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully!")
print(ranked[["baseline_score", "reason_code", "action"]].head())

CSV saved successfully!
       baseline_score                     reason_code           action
6653         0.560399  REFRESH_STALE_HIGH_IMPRESSIONS  REFRESH_CONTENT
19636        0.533037  REFRESH_STALE_HIGH_IMPRESSIONS  REFRESH_CONTENT
29400        0.517692  REFRESH_STALE_HIGH_IMPRESSIONS  REFRESH_CONTENT
17812        0.517198  REFRESH_STALE_HIGH_IMPRESSIONS  REFRESH_CONTENT
26844        0.504982  REFRESH_STALE_HIGH_IMPRESSIONS  REFRESH_CONTENT


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
top20 = ranked.head(20).reset_index(drop=True)

for i, row in top20.iterrows():

    print("=" * 80)
    print(f"Rank {i+1}")

    print(f"Action: {row['action']}")
    print(f"Reason Code: {row['reason_code']}")

    # Confidence Note
    if row["baseline_score"] >= 0.80:
        confidence = "High confidence"
    elif row["baseline_score"] >= 0.60:
        confidence = "Medium confidence"
    else:
        confidence = "Low confidence"

    print(f"Confidence Note: {confidence}")

    # Why selected
    print(
        f"Why selected: High baseline score ({row['baseline_score']:.3f}) "
        f"based on impressions, average position, and days since last update."
    )

    # What would make it wrong
    print(
        "What would make it wrong: The page may already satisfy user intent, "
        "its traffic may be seasonal, or poor ranking could be caused by strong "
        "competition rather than outdated content."
    )

    print()

Rank 1
Action: REFRESH_CONTENT
Reason Code: REFRESH_STALE_HIGH_IMPRESSIONS
Confidence Note: Low confidence
Why selected: High baseline score (0.560) based on impressions, average position, and days since last update.
What would make it wrong: The page may already satisfy user intent, its traffic may be seasonal, or poor ranking could be caused by strong competition rather than outdated content.

Rank 2
Action: REFRESH_CONTENT
Reason Code: REFRESH_STALE_HIGH_IMPRESSIONS
Confidence Note: Low confidence
Why selected: High baseline score (0.533) based on impressions, average position, and days since last update.
What would make it wrong: The page may already satisfy user intent, its traffic may be seasonal, or poor ranking could be caused by strong competition rather than outdated content.

Rank 3
Action: REFRESH_CONTENT
Reason Code: REFRESH_STALE_HIGH_IMPRESSIONS
Confidence Note: Low confidence
Why selected: High baseline score (0.518) based on impressions, average position, and days sinc

## 4. Weak picks + leakage check

# Weak Picks

The baseline rule is simple and transparent, so some content items may receive high scores even if they are not the best candidates for refresh.

Possible weak picks include:

- Pages with high impressions but already accurate and up-to-date content.
- Seasonal pages whose traffic naturally changes throughout the year.
- Pages with poor ranking because of strong competition instead of outdated content.
- Pages with high impressions but low business value.

Similarly, some good candidates may receive lower scores because they have low impressions even though updating them could improve their performance.

---

# Leakage Check

The baseline rule uses only current observable information:

- impressions_90d
- avg_position
- days_since_last_update

The rule does **not** use:

- trend_direction
- trend_pct
- any future performance metrics
- any label-derived variables
- product flags
- future time windows

Therefore, the baseline does not contain data leakage and can serve as a fair benchmark for the machine learning model in Week 5.

In [11]:
used_features = [
    "impressions_90d",
    "avg_position",
    "days_since_last_update"
]

print("Features used:", used_features)
print("No future-window features used.")
print("No label-derived features used.")
print("No product flags used.")

Features used: ['impressions_90d', 'avg_position', 'days_since_last_update']
No future-window features used.
No label-derived features used.
No product flags used.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.